# SRKR Notebook

Runs a one-dimensional spatial scan in corrected `x_cor_um` or `y_cor_um` coordinates.

Review `config_path`, then connect devices explicitly; this notebook does not auto-connect hardware.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from kohdalab.api import Experiment, format_point, load_config, make_srkr_live_update, srkr_plan


config_path: Path | None = None  # Set a lab-specific JSON path before hardware use.
config = load_config() if config_path is None else load_config(config_path)
experiment = Experiment(config, auto_connect=False)
# Review config, then run experiment.connect_all() before acquiring data.
zero = config["measurements"]["move_abs"]["zero"]

def print_status(status: str) -> None:
    print(f"status: {status}", flush=True)


In [ ]:
fast_axis = "x"  # "x" or "y"
minimum_um = -2.0
maximum_um = 2.0
step_um = 2.0
wait_s = 0.0
return_to_zero = True
output = None  # None uses the output path configured for srkr.
y_key = "R_V"

plan = srkr_plan(
    axis=fast_axis,
    minimum_um=minimum_um,
    maximum_um=maximum_um,
    step_um=step_um,
    zero_by_axis={"x": float(zero.get("x_um", 0.0)), "y": float(zero.get("y_um", 0.0))},
    coordinate="measurement",
)
live_update = make_srkr_live_update(fast_axis=plan.axis, y_key=y_key, title=f"SRKR {plan.axis.upper()}")
axis_key = f"{plan.axis}_cor_um"


In [ ]:
def handle_point(point):
    print(format_point(point, axis_key=axis_key), flush=True)
    live_update(point)


rows = experiment.run_srkr(
    plan=plan,
    wait_s=wait_s,
    return_to_zero=return_to_zero,
    output=output,
    on_status=print_status,
    on_point=handle_point,
)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
